In [2]:
import numpy as np
import random

random.seed(42)
np.random.seed(42)

days, initial_price = 100, 100
prices = initial_price * np.cumprod(1 + np.random.normal(0.001, 0.02, days))

POPULATION_SIZE, GENERATIONS = 20, 10
TRANSACTION_FEE, DRAWDOWN_PENALTY = 0.001, 0.5

SHORT_CANDIDATES = list(range(2, 16))
LONG_CANDIDATES = list(range(16, 41))

ALPHA, BETA, RHO, Q = 1.0, 2.0, 0.1, 1.0
tau_short = {s: 1.0 for s in SHORT_CANDIDATES}
tau_long = {l: 1.0 for l in LONG_CANDIDATES}


def evaluate(strategy):
    short, long = strategy
    s_ma = np.convolve(prices, np.ones(short) / short, mode="full")[:days]
    l_ma = np.convolve(prices, np.ones(long) / long, mode="full")[:days]

    position = np.roll((s_ma > l_ma).astype(float), 1)
    position[0] = 0
    trades = np.abs(np.diff(position, prepend=0))

    mkt_ret = np.zeros(days)
    mkt_ret[1:] = prices[1:] / prices[:-1] - 1
    strat_ret = position * mkt_ret - trades * TRANSACTION_FEE

    portfolio = np.cumprod(1 + strat_ret)
    net_profit = (portfolio[-1] - 1) * 100
    peak = np.maximum.accumulate(portfolio)
    max_drawdown = np.max((peak - portfolio) / peak) * 100
    total_fees = np.sum(trades * TRANSACTION_FEE) * 100

    return net_profit - DRAWDOWN_PENALTY * max_drawdown, net_profit, max_drawdown, total_fees


eta_short = {s: max(0.01, np.mean([evaluate([s, l])[0] for l in LONG_CANDIDATES]) + 100) for s in SHORT_CANDIDATES}
eta_long = {l: max(0.01, np.mean([evaluate([s, l])[0] for s in SHORT_CANDIDATES]) + 100) for l in LONG_CANDIDATES}


def select_param(candidates, tau, eta):
    w = [(tau[c] ** ALPHA) * (eta[c] ** BETA) for c in candidates]
    return np.random.choice(candidates, p=np.array(w) / sum(w))


best_strat, best_fit = None, -float("inf")

for _ in range(GENERATIONS):
    ants = []
    for _ in range(POPULATION_SIZE):
        s = select_param(SHORT_CANDIDATES, tau_short, eta_short)
        l = select_param(LONG_CANDIDATES, tau_long, eta_long)
        ants.append(([s, l], *evaluate([s, l])))

    ants.sort(key=lambda x: x[1], reverse=True)
    if ants[0][1] > best_fit:
        best_fit, best_strat = ants[0][1], ants[0][0].copy()

    for s in SHORT_CANDIDATES: tau_short[s] *= (1.0 - RHO)
    for l in LONG_CANDIDATES: tau_long[l] *= (1.0 - RHO)

    for strat, fit, _, _, _ in ants:
        dep = Q * max(0.01, fit + 100) / 100.0
        tau_short[strat[0]] += dep
        tau_long[strat[1]] += dep

fit, profit, dd, fees = evaluate(best_strat)

print(f"Best Strategy: Short={best_strat[0]}, Long={best_strat[1]}")
print(f"Profit: {profit:.2f}% | Drawdown: {dd:.2f}% | Fees: {fees:.2f}% | Fitness: {fit:.2f}")

Best Strategy: Short=11, Long=23
Profit: 5.19% | Drawdown: 13.95% | Fees: 0.50% | Fitness: -1.78
